# Notebook 3b: Learning-rate sensitivity (M4a)

Quick: reads `results_optimizers.json` from notebook 03 and plots how each optimizer's validation AUC depends on its learning rate. No new training. Output figure goes into the paper's Discussion section.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SPLITS_DIR = '/content/drive/MyDrive/ECE567_Final/splits'
with open(os.path.join(SPLITS_DIR, 'results_optimizers.json')) as fh:
    res = json.load(fh)

sweep = pd.DataFrame(res['sweep'])
lbfgs = res['config']['lbfgs_ref_auc']
print('rows:', len(sweep))
sweep.head()

### Per-optimizer lr response (small multiples)

In [ ]:
opt_order = ['SGD', 'Momentum', 'NAG', 'AdaGrad', 'RMSprop', 'Adam', 'AdamW']

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=True)
axes = axes.flatten()
for ax, name in zip(axes, opt_order):
    sub = sweep[sweep.optimizer == name].sort_values('lr')
    ax.plot(sub.lr, sub.best_auc, 'o-', color='steelblue')
    ax.axhline(lbfgs, ls='--', color='black', alpha=0.5, label='L-BFGS')
    ax.set_xscale('log')
    ax.set_title(name)
    ax.set_xlabel('learning rate')
    ax.grid(alpha=0.3)
    # mark the winning lr
    if not sub.best_auc.isna().all():
        best_idx = sub.best_auc.idxmax()
        ax.plot(sub.loc[best_idx, 'lr'], sub.loc[best_idx, 'best_auc'],
                marker='*', markersize=18, color='crimson', zorder=5)
axes[0].set_ylabel('best val AUC (20 epochs)')
axes[4].set_ylabel('best val AUC (20 epochs)')
# hide the unused 8th panel
axes[-1].axis('off')
axes[-1].legend(*axes[0].get_legend_handles_labels(), loc='center', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(SPLITS_DIR, 'fig_lr_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()

### Robustness ranking — width of the "good" lr band

How wide an lr range keeps each optimizer within 0.001 AUC of its own best? A wider band means less tuning sensitivity.

In [ ]:
rows = []
for name in opt_order:
    sub = sweep[sweep.optimizer == name].dropna(subset=['best_auc']).sort_values('lr')
    if sub.empty:
        continue
    best = sub.best_auc.max()
    good = sub[sub.best_auc >= best - 1e-3]
    rows.append({
        'optimizer': name,
        'best_auc' : best,
        'best_lr'  : sub.loc[sub.best_auc.idxmax(), 'lr'],
        'n_good_lrs': len(good),
        'lr_range_good': f'{good.lr.min():g}–{good.lr.max():g}' if len(good) > 0 else 'none',
        'lr_ratio_good': good.lr.max() / good.lr.min() if len(good) > 1 else 1.0,
    })
robust = pd.DataFrame(rows).sort_values('lr_ratio_good', ascending=False)
print(robust.to_string(index=False))
robust.to_csv(os.path.join(SPLITS_DIR, 'results_lr_robustness.csv'), index=False)